# Part 4 supplementary notebook — frozen fusion-results review

> **Primary reproducible notebook:** `MemberC_Phases_4_8_Full_Reproduction.ipynb`

This small notebook is intentionally read-only. It loads the already frozen Part 4 artifacts and summarizes them without retraining, changing thresholds, or reopening model-selection decisions.

The code cells answer five review questions:

1. Which candidate was frozen, and did validation/test meet the internal recall gate?
2. What are the continuous and aggregate class metrics?
3. Which candidate families were compared?
4. Where do per-class precision/recall/F1 errors occur?
5. Which features dominate CatBoost importance and validation SHAP?

This separation is useful because **results review should not silently modify the experiment**.


In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path('..').resolve()
metrics = json.loads((ROOT/'outputs/final_metrics.json').read_text(encoding='utf-8'))
metrics['selected_candidate'], metrics['validation_gate'], metrics['test_gate']

('hybrid_catboost_rmse_depth3', True, False)

In [2]:
summary = pd.DataFrame([
    {'split':'Validation', **metrics['validation']['continuous'], 'accuracy':metrics['validation']['classification']['accuracy'], 'macro_f1':metrics['validation']['classification']['macro_f1']},
    {'split':'Test', **metrics['test']['continuous'], 'accuracy':metrics['test']['classification']['accuracy'], 'macro_f1':metrics['test']['classification']['macro_f1']},
])
summary

,split,mae,rmse,pearson,spearman,prediction_min,prediction_max,accuracy,macro_f1
0,Validation,0.834757,1.133565,0.857657,0.818490,1.0,7.756248,0.756637,0.671679
1,Test,0.917356,1.245912,0.833266,0.790059,1.0,7.676382,0.752759,0.628420


In [3]:
pd.read_csv(ROOT/'outputs/candidate_leaderboard.csv').head(12)

,candidate,feature_profile,validation_mae,validation_rmse,validation_pearson,threshold_status,feasible_threshold_count,threshold_low_moderate,threshold_moderate_high,threshold_high_very_high,validation_accuracy,validation_macro_f1,validation_low_recall,validation_moderate_recall,validation_high_recall,validation_very_high_recall,validation_low_precision,validation_moderate_precision,validation_high_precision,validation_very_high_precision,selected
0,hybrid_catboost_rmse_depth3,hybrid,0.834757,1.133565,0.857657,feasible,4485,3.095439,4.905161,6.462135,0.756637,0.671679,0.845118,0.582090,0.523810,0.76,0.919414,0.382353,0.622642,0.791667,True
1,hybrid_catboost_rmse_depth4,hybrid,0.835160,1.133484,0.857165,feasible,2222,2.597060,4.534750,6.411443,0.723451,0.653317,0.777778,0.552239,0.634921,0.76,0.950617,0.327434,0.571429,0.730769,False
2,weighted_blend_member2_0.80,scalar,0.842190,1.125543,0.864515,feasible,4750,3.068673,4.649734,6.387064,0.765487,0.674515,0.845118,0.567164,0.587302,0.80,0.936567,0.404255,0.606557,0.689655,False
3,weighted_blend_member2_0.95,scalar,0.817628,1.134044,0.862292,feasible,9682,2.959015,4.819307,6.706354,0.763274,0.671669,0.841751,0.582090,0.571429,0.80,0.943396,0.402062,0.590164,0.689655,False
4,member2_only,scalar,0.814541,1.144482,0.860464,feasible,10873,3.172451,4.886269,6.757587,0.772124,0.665355,0.875421,0.522388,0.539683,0.80,0.931900,0.411765,0.576271,0.689655,False
5,weighted_blend_member2_1.00,scalar,0.814541,1.144482,0.860464,feasible,10873,3.172451,4.886269,6.757587,0.772124,0.665355,0.875421,0.522388,0.539683,0.80,0.931900,0.411765,0.576271,0.689655,False
6,scalar_catboost_depth3,scalar,0.822555,1.128538,0.858277,feasible,4675,3.027015,4.747170,6.213054,0.761062,0.666246,0.841751,0.611940,0.539683,0.76,0.936330,0.401961,0.618182,0.678571,False
7,weighted_blend_member2_0.90,scalar,0.822549,1.127340,0.863629,feasible,7864,3.200225,4.766135,6.637161,0.774336,0.663889,0.878788,0.507463,0.571429,0.76,0.928826,0.419753,0.580645,0.678571,False
8,hybrid_catboost_mae_depth4,hybrid,0.734985,1.099061,0.864075,feasible,839,2.616444,4.251138,6.132085,0.741150,0.634455,0.841751,0.507463,0.507937,0.76,0.932836,0.357895,0.524590,0.678571,False
9,weighted_blend_member2_0.85,scalar,0.831564,1.124453,0.864423,feasible,5494,3.167385,4.680988,6.413631,0.774336,0.676846,0.865320,0.522388,0.587302,0.84,0.931159,0.416667,0.606557,0.677419,False


In [4]:
pd.concat([
    pd.read_csv(ROOT/'outputs/validation_per_class.csv').assign(split='Validation'),
    pd.read_csv(ROOT/'outputs/test_per_class.csv').assign(split='Test'),
], ignore_index=True)

,clinical_class,precision,recall,f1,support,recall_ci_low,recall_ci_high,minimum_required_recall,split
0,Low,0.919414,0.845118,0.880702,297,0.799592,0.881830,0.75,Validation
1,Moderate,0.382353,0.582090,0.461538,67,0.462699,0.692577,0.50,Validation
2,High,0.622642,0.523810,0.568966,63,0.402704,0.642179,0.50,Validation
3,Very High,0.791667,0.760000,0.775510,25,0.565703,0.885037,0.75,Validation
4,Low,0.915493,0.875421,0.895009,297,0.833003,0.908251,0.75,Test
5,Moderate,0.354430,0.417910,0.383562,67,0.307423,0.537301,0.50,Test
6,High,0.568966,0.523810,0.545455,63,0.402704,0.642179,0.50,Test
7,Very High,0.625000,0.769231,0.689655,26,0.579484,0.889662,0.75,Test


In [5]:
pd.read_csv(ROOT/'outputs/global_feature_importance.csv').head(20)

,feature,importance
0,base_mean,42.426698
1,member2_prediction,41.315227
2,member1_prediction,2.983195
3,post_pos_emoji,2.004879
4,post_neg_emoji,1.795264
5,base_difference,0.939318
6,post_emoji_per_100_words,0.580484
7,post_temp_neg_ratio,0.488968
8,sub_category,0.384120
9,post_neg_count,0.349168


In [6]:
pd.read_csv(ROOT/'outputs/validation_shap_summary.csv').head(20)

,feature,mean_absolute_shap,mean_signed_shap
0,member2_prediction,0.761263,-0.147580
1,base_mean,0.731600,-0.201375
2,member1_prediction,0.080779,-0.026855
3,post_neg_emoji,0.052867,-0.011649
4,post_pos_emoji,0.051852,0.007152
5,base_difference,0.026638,0.000971
6,post_temp_neg_ratio,0.025068,-0.003446
7,has_reply,0.020471,0.004666
8,post_neg_count,0.017995,-0.003300
9,post_neg_word_ratio,0.016979,-0.000074


## Final result interpretation

Validation met all four internal recall targets. The locked test met **3 of 4**:

- Low: pass
- Moderate: **41.79% recall, below the 50% internal target**
- High: pass
- Very High: pass

The correct project status is therefore **3_of_4_internal_recall_targets_met**.

This does **not** automatically imply “research/shadow only.” The separate deployment interpretation is **candidate for a limited human-in-the-loop risk-monitoring overlay**, subject to privacy, governance, workload evaluation, and human review. It is not a clinical diagnostic or autonomous emergency system.

Because the test is already opened, this notebook must remain descriptive; it must not be used to choose a new threshold/model.
